In [ ]:
# Install compatible PyTorch
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

# Install xformers compatible with torch 2.3
!pip install xformers==0.0.27

# Install unsloth
!pip install unsloth

40

In [3]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [4]:
from datasets import load_dataset  ,concatenate_datasets 
import random  

SEED = 42 
TARGET_DATASET = 100000
def get_load_dataset(): 
    ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
    ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
    ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
    ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train") 
    return ds1 , ds2 , ds3 , ds4 


## applying the weights of dataset 
WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
} 

def weights_of_dataset(ds1,ds2,ds3,ds4): 
    random.seed(SEED)

    source_map = {
        "instruction":  ds1,
        "summarization": ds2,
        "extraction":   ds3,
        "decomposition": ds4,
    } 
    selected_split = [] 
    for name ,ds in source_map.items(): 
        target_n = int(TARGET_DATASET * WEIGHTS[name]) 
        
        if len(ds) >= target_n:
            idx =  random.sample(range(len(ds)) , target_n) 
            split = ds.select(idx) 
            
        else: 
            repeats = (target_n // len(ds)) + 1
            idx = (list(range(len(ds))) * repeats)[:target_n]
            random.shuffle(idx)
            split = ds.select(idx) 
            
        selected_split.append(split) 
        
    combined_dataset = concatenate_datasets(selected_split) 
    combined = combined_dataset.shuffle(seed=SEED) 
    return combined 



In [5]:
import pandas as pd

DS1 , DS2 , DS3 , DS4 = get_load_dataset() 

DATASET = weights_of_dataset(DS1 , DS2 , DS3 , DS4) 

ds = pd.DataFrame(DATASET) 

In [6]:
ds['task'].value_counts()

task
instruction_tune          35000
summarization             30000
information_extraction    20000
query_decomposition       15000
Name: count, dtype: int64

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [7]:
model = FastLanguageModel.get_peft_model(
    model , 
    finetune_vision_layers = False , 
    finetune_language_layers = True , # as you know what we focus on **only text**
    finetune_attention_modules=True , 
    finetune_mlp_modules = True , 

    r = 16   ,  ## if it is larger high accuracy but might overfit 
    lora_alpha = 16  , # r=lora-alpha 
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [8]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 768)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-11): 12 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear(in_features=768, out_features=2304, bias=True)
                (proj): Linear(in_features=768, out_features=768, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
                (linear

In [9]:
ds.head()

,dataset,task,system,messages
0,cnn_dailymail,summarization,You are a research summarization assistant. Yo...,"[{'role': 'user', 'content': 'Provide a dense,..."
1,cnn_dailymail,summarization,You are a research summarization assistant. Yo...,"[{'role': 'user', 'content': 'Write a brief, a..."
2,alpaca,instruction_tune,You are Helpful assistant.,"[{'role': 'user', 'content': 'Name some instit..."
3,synthetic_decomposition,query_decomposition,You are a research planning agent. Break compl...,"[{'role': 'user', 'content': 'Decompose this r..."
4,synthetic_decomposition,query_decomposition,You are a research planning agent. Break compl...,"[{'role': 'user', 'content': 'Decompose this r..."


In [10]:
def format_to_chatml(sample):
    system   = (sample.get("system") or "You are a helpful AI assistant.").strip()
    messages = sample.get("messages") or []

    parts = [f"<|im_start|>system\n{system}<|im_end|>"]
    for msg in messages:
        role    = (msg.get("role")    or "").strip()
        content = (msg.get("content") or "").strip()
        if role in ("user", "assistant") and content:
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")

    return {"text": "\n".join(parts) + "\n"}  


In [5]:
from huggingface_hub import login
login(token="") 

from huggingface_hub import whoami

try:
    user_info = whoami()
    print(f"Logged in as: {user_info['name']}")
except Exception as e:
    print(f"Not logged in. Error: {e}")


LocalProtocolError: Illegal header value b'Bearer '

In [4]:
"""
Speed-optimized fine-tuning for Kaggle T4
==========================================
Changes from v4 (float32, no packing, 30k):
  1. fp16=True  with proper bitsandbytes fix  → 2.5x faster compute
  2. packing=True                              → 1.5x faster (no wasted padding tokens)
  3. TARGET_TOTAL = 20_000                     → less data but same quality (see note)
  4. MAX_SEQ_LENGTH = 1024  (was 512)          → packing needs longer context to fill
  5. BATCH_SIZE = 2  (was 1)                   → T4 can handle 2 with packing+fp16
  6. EPOCHS = 1  (was 2)                       → 1 epoch on 20k ≈ 2 epochs on 10k quality-wise

Combined speedup: ~3.5x  →  7hrs becomes ~2hrs

WHY fp16 works now but broke before:
  The CUBLAS error was caused by bitsandbytes computing in fp16 BUT the model
  weights loaded as fp16 too — causing a dtype mismatch in the gemm kernel.
  Fix: load model as float32, but tell bitsandbytes to compute in float16.
  The dequantized weights get cast to fp16 at runtime, activations stay fp16.
  This is the standard QLoRA setup and works correctly on T4.
"""

import os, gc, random
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

os.environ["WANDB_DISABLED"]         = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen3-0.6B"
MAX_SEQ_LENGTH = 1024          # longer = packing fills context better = faster
LORA_RANK      = 8
BATCH_SIZE     = 2             # safe with fp16 + packing on T4
GRAD_ACCUM     = 8             # effective batch = 2 * 8 = 16 (same as before)
LEARNING_RATE  = 2e-4
EPOCHS         = 1             # 1 epoch on 20k is enough — model sees each sample once
OUTPUT_DIR     = "/kaggle/working/qwen3-research-agent"
SEED           = 42
TARGET_TOTAL   = 20_000        # 20k is sufficient — quality >> quantity for SLMs

WEIGHTS = {
    "instruction":   0.35,
    "summarization": 0.30,
    "extraction":    0.20,
    "decomposition": 0.15,
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD MODEL
# ─────────────────────────────────────────────────────────────────────────────
print("[ 1/5 ] Loading model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code = True,
    padding_side      = "right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,   # fp16 compute — fast on T4
    bnb_4bit_use_double_quant = True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = {"": 0},
    trust_remote_code   = True,
    torch_dtype         = torch.float32,         # load in float32 — avoids CUBLAS mismatch
)                                                # bitsandbytes casts to fp16 at compute time

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing = True,
)

lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = LORA_RANK,
    lora_alpha     = LORA_RANK * 2,
    lora_dropout   = 0.05,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

torch.cuda.empty_cache()
gc.collect()
print("[ 1/5 ] Model ready ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — LOAD + COMBINE DATASETS
# ─────────────────────────────────────────────────────────────────────────────
print("[ 2/5 ] Loading datasets...")

ds1 = load_dataset("Rohit-Katkar2003/Instruction-dataset",   split="train")
ds2 = load_dataset("Rohit-Katkar2003/summarization-dataset", split="train")
ds3 = load_dataset("Rohit-Katkar2003/Extractional-dataset",  split="train")
ds4 = load_dataset("Rohit-Katkar2003/decomposition-dataset", split="train")

print(f"    D1={len(ds1):,}  D2={len(ds2):,}  D3={len(ds3):,}  D4={len(ds4):,}")

random.seed(SEED)
splits = []
for name, ds, w in [
    ("instruction",   ds1, WEIGHTS["instruction"]),
    ("summarization", ds2, WEIGHTS["summarization"]),
    ("extraction",    ds3, WEIGHTS["extraction"]),
    ("decomposition", ds4, WEIGHTS["decomposition"]),
]:
    target_n = int(TARGET_TOTAL * w)
    if len(ds) >= target_n:
        idx = random.sample(range(len(ds)), target_n)
    else:
        repeats = (target_n // len(ds)) + 1
        idx     = (list(range(len(ds))) * repeats)[:target_n]
        random.shuffle(idx)
        print(f"    Upsampled {name}: {len(ds)} → {target_n}")
    splits.append(ds.select(idx))

combined = concatenate_datasets(splits).shuffle(seed=SEED)
del ds1, ds2, ds3, ds4
gc.collect()
print(f"[ 2/5 ] Combined: {len(combined):,} samples ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — FORMAT TO CHATML
# ─────────────────────────────────────────────────────────────────────────────
print("[ 3/5 ] Formatting to ChatML...")

def format_to_chatml(sample):
    system   = (sample.get("system") or "You are a helpful AI assistant.").strip()
    messages = sample.get("messages") or []
    parts    = [f"<|im_start|>system\n{system}<|im_end|>"]
    for msg in messages:
        role    = (msg.get("role")    or "").strip()
        content = (msg.get("content") or "").strip()
        if role in ("user", "assistant") and content:
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    return {"text": "\n".join(parts) + "\n"}

formatted = combined.map(
    format_to_chatml,
    remove_columns = combined.column_names,
    num_proc       = 1,
)
# For packing: keep sequences under MAX_SEQ_LENGTH
# packing will concatenate short sequences to fill the context window
formatted = formatted.filter(lambda x: 50 < len(x["text"]) <= MAX_SEQ_LENGTH * 3)
del combined
gc.collect()

split    = formatted.train_test_split(test_size=0.05, seed=SEED)
train_ds = split["train"]
val_ds   = split["test"]
del formatted
gc.collect()

print(f"[ 3/5 ] Train: {len(train_ds):,}  Val: {len(val_ds):,} ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — TRAIN
# ─────────────────────────────────────────────────────────────────────────────
print("[ 4/5 ] Starting training...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,

    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,

    bf16                        = False,
    fp16                        = True,          # ✅ fp16 training (2.5x faster than float32)

    packing                     = True,          # ✅ pack short sequences → no wasted padding
    # max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",

    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,

    logging_steps               = 25,
    report_to                   = "none",

    seed                        = SEED,
    dataloader_num_workers      = 0,
    optim                       = "paged_adamw_8bit",
)

trainer = SFTTrainer(
    model         = model,
    # tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args          = args,
)

trainer.train()
print("[ 4/5 ] Training complete ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — SAVE
# ─────────────────────────────────────────────────────────────────────────────
print("[ 5/5 ] Saving model...")

adapter_path = f"{OUTPUT_DIR}/lora-adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"[ 5/5 ] Saved → {adapter_path} ✓")
print("=" * 40)
print(f"Estimated time: ~1.5-2 hrs on T4")
print("DONE!")
print("=" * 40)

[ 1/5 ] Loading model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 5,046,272 || all params: 756,678,656 || trainable%: 0.6669
[ 1/5 ] Model ready ✓
[ 2/5 ] Loading datasets...
    D1=71,179  D2=100,000  D3=100,000  D4=500
    Upsampled decomposition: 500 → 3000
[ 2/5 ] Combined: 20,000 samples ✓
[ 3/5 ] Formatting to ChatML...


Map (num_proc=1):   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/992 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/992 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 